# 03) Baseline Models and Feature Selection

In [1]:
import json

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

train = pd.read_csv("data/train.csv", parse_dates=["date"])
with open("data/columns.json") as f:
    columns = json.load(f)

TARGET_COL = columns["target_cols"][0]
FEATURE_COLS = columns["feature_cols"]

print(f"train: {train.shape[0]} rows, {len(FEATURE_COLS)} features, target = {TARGET_COL!r}")

train: 8443 rows, 33 features, target = 'target_bps_vs_mid'


## Expanding-window cross-validation split

A single fixed fit/eval split gives one point estimate of eval MSE, entirely dependent on which
handful of days happened to land in eval -- not something to pick a penalty on with confidence.
Expanding-window CV rolls the eval block forward across multiple folds: each fold trains on every
day strictly before its eval block (never after -- no lookahead) and the training window grows as
the boundary moves forward. `data/test.csv` is still untouched; all folds live inside
`data/train.csv`.

Folds are cut on day boundaries via `TimeSeriesSplit` applied to the sorted unique dates, not to
rows directly, so a day's rows never land on both sides of a fold.

In [2]:
unique_dates = np.sort(train["date"].unique())
tscv = TimeSeriesSplit(n_splits=5)

folds = []
for fold_idx, (train_pos, eval_pos) in enumerate(tscv.split(unique_dates)):
    fit_dates = unique_dates[train_pos]
    eval_dates = unique_dates[eval_pos]
    assert fit_dates.max() < eval_dates.min(), "eval dates must all fall after fit dates"

    fit_mask = train["date"].isin(fit_dates)
    eval_mask = train["date"].isin(eval_dates)
    folds.append((fit_mask, eval_mask))

    print(f"fold {fold_idx}: fit {fit_mask.sum():>4} rows ({len(fit_dates)} days, "
          f"{pd.Timestamp(fit_dates.min()).date()} to {pd.Timestamp(fit_dates.max()).date()}) | "
          f"eval {eval_mask.sum():>4} rows ({len(eval_dates)} days, "
          f"{pd.Timestamp(eval_dates.min()).date()} to {pd.Timestamp(eval_dates.max()).date()})")

fold 0: fit 3476 rows (7 days, 2025-03-03 to 2025-03-11) | eval  993 rows (2 days, 2025-03-12 to 2025-03-13)
fold 1: fit 4469 rows (9 days, 2025-03-03 to 2025-03-13) | eval  994 rows (2 days, 2025-03-14 to 2025-03-17)
fold 2: fit 5463 rows (11 days, 2025-03-03 to 2025-03-17) | eval  993 rows (2 days, 2025-03-18 to 2025-03-19)
fold 3: fit 6456 rows (13 days, 2025-03-03 to 2025-03-19) | eval  993 rows (2 days, 2025-03-20 to 2025-03-21)
fold 4: fit 7449 rows (15 days, 2025-03-03 to 2025-03-21) | eval  994 rows (2 days, 2025-03-24 to 2025-03-25)


## Ridge regression, tuned over the expanding-window folds

Same pipeline as before -- median-impute (only `near_far_spread_bps` has nulls) -> standardize ->
Ridge, both fit on each fold's `fit` side only. Same alpha grid, wide enough to bracket where eval
MSE bottoms out. For each alpha, fit/eval MSE is computed per fold, then averaged across folds;
`eval_mse_std` shows how much that average is actually worth trusting.

In [3]:
alphas = [0.1, 1.0, 10.0, 100.0, 1_000.0, 10_000.0, 100_000.0, 1_000_000.0]
results = []

for alpha in alphas:
    fit_mses, eval_mses = [], []
    for fit_mask, eval_mask in folds:
        X_fit, y_fit = train.loc[fit_mask, FEATURE_COLS], train.loc[fit_mask, TARGET_COL]
        X_eval, y_eval = train.loc[eval_mask, FEATURE_COLS], train.loc[eval_mask, TARGET_COL]

        model = Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
            ("ridge", Ridge(alpha=alpha)),
        ])
        model.fit(X_fit, y_fit)

        fit_mses.append(mean_squared_error(y_fit, model.predict(X_fit)))
        eval_mses.append(mean_squared_error(y_eval, model.predict(X_eval)))

    results.append({
        "alpha": alpha,
        "fit_mse_mean": np.mean(fit_mses),
        "eval_mse_mean": np.mean(eval_mses),
        "eval_mse_std": np.std(eval_mses),
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

    alpha  fit_mse_mean  eval_mse_mean  eval_mse_std
      0.1     41.348672      52.162516     21.326278
      1.0     41.410435      52.148261     21.359348
     10.0     41.434449      52.109778     21.329444
    100.0     41.463279      51.720591     20.924420
   1000.0     42.305508      49.676513     19.048149
  10000.0     51.564921      51.451846     26.968095
 100000.0     68.716971      66.471754     39.694704
1000000.0     73.364203      71.408343     42.933959
